## Setup and Imports

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import plotly.io as pio

#formatting command so tht pandas doesn't truncate (shortens the data to preserve memory) 
pd.set_option('display.max_columns', None)

print("Environment setup complete.")
pio.renderers.default = "iframe"

Environment setup complete.


In [3]:
df=pd.read_csv('online_retail.csv', encoding='ISO-8859-1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
print(f"Dataset Shape: {df.shape}\n") #Tells rows and columns
print("Column Info")
df.info() #Tells data type and memory usage
print("Numberical Summary")
df.describe() #Gives mean, min, max, standard deviation

Dataset Shape: (541909, 8)

Column Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
Numberical Summary


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## Data Cleaning (Missing values)

In [5]:
#Check for missing or null values per column
missing_data=df.isnull().sum()
print("Missing values per column:")
print(missing_data)

Missing values per column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [6]:
#Drop rows where CustomerID is null and modify the dataframe in place
df_clean=df.dropna(subset=['CustomerID']).copy()

print(f"Rows remaining after dropping missing IDs: {df_clean.shape[0]}")

Rows remaining after dropping missing IDs: 406829


## Isolate Outliers using IQR (Interquartile Range)

In [7]:

df_clean['TotalSales']=df_clean['Quantity']*df_clean['UnitPrice']

'''
To calculate bounds we use below:
IQR = Q3 - Q1
Lower Bound = Q1 - (1.5 * IQR)
Upper Bound = Q3 + (1.5 * IQR)

(any data point fallin oustide these boundaries is considered an outlier)
'''

# Calculate the 25th (Q1) and 75th (Q3) percentiles
Q1 = df_clean['TotalSales'].quantile(0.25)
Q3 = df_clean['TotalSales'].quantile(0.75)
IQR = Q3 - Q1

#Boundary walls definition
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

#Separate standard transactions from the extreme anomalies
standard_sales = df_clean[(df_clean['TotalSales']>=lower_bound) & (df_clean['TotalSales']<=upper_bound)]
outliers = df_clean[(df_clean['TotalSales']<lower_bound) | (df_clean['TotalSales']>upper_bound)]

print(f"Standard Transactions: {standard_sales.shape[0]} rows")
print(f"Anomalous Transactions: {outliers.shape[0]} rows")


Standard Transactions: 373649 rows
Anomalous Transactions: 33180 rows


## Aggregating Data for Month-Over-Month (MoM) growth

In [8]:

# An explicit copy to avoid SettingWithCopyWarning
standard_sales = standard_sales.copy()

# 1. Convert InvoiceDate to a true Datetime object
standard_sales['InvoiceDate'] = pd.to_datetime(standard_sales['InvoiceDate'])

# 2. Extract Year and Month (e.g., '2011-11') so we can group by it
standard_sales['YearMonth'] = standard_sales['InvoiceDate'].dt.to_period('M')

# 3. Group by month and sum the sales
monthly_trends = standard_sales.groupby('YearMonth')['TotalSales'].sum().reset_index()

# 4. Convert YearMonth back to a string format
monthly_trends['YearMonth'] = monthly_trends['YearMonth'].astype(str)

# 5. Calculate Month-over-Month growth percentage
monthly_trends['MoM_Growth_Pct'] = monthly_trends['TotalSales'].pct_change() * 100

monthly_trends


,YearMonth,TotalSales,MoM_Growth_Pct
0,2010-12,276676.050,NaN
1,2011-01,236248.140,-14.612002
2,2011-02,224825.800,-4.834891
3,2011-03,297096.890,32.145372
4,2011-04,256216.991,-13.759787
5,2011-05,334938.850,30.724683
6,2011-06,303004.100,-9.534502
7,2011-07,292205.831,-3.563737
8,2011-08,315291.130,7.900355
9,2011-09,470518.242,49.232946


## Visualizations

In [9]:
!pip install -U kaleido

In [10]:
# Group data by country to see where revenue originates
country_data = standard_sales.groupby('Country')['TotalSales'].sum().reset_index()

# Build an interactive choropleth world map
fig_map = px.choropleth(
    country_data, 
    locations="Country", 
    locationmode="country names",
    color="TotalSales", 
    title="Global Revenue Distribution (Interactive Map)",
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={'TotalSales':'Total Revenue ($)'}
)


fig_map.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig_map.show()

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [ ]:
# Relationship between unit price and the quantity of customers purchase.


scatter_sample = standard_sales.sample(n=min(5000, len(standard_sales)), random_state=42)

fig_scatter = px.scatter(
    scatter_sample, 
    x="UnitPrice", 
    y="Quantity", 
    color="Country",
    title="Customer Purchase Behavior: Quantity vs Unit Price",
    hover_data=['Description'],
    opacity=0.7
)

fig_scatter.show()